<h2>Netflix_Project</h2>

### Problem Statement
For the purpose of this Project we will be creating one such Recommendation Engine from the ground-up, where every single user, based on there area of interest and ratings, would be recommended a list of movies that are best suited for them.
1. Find out the list of most popular and liked genre
2. Create Model that finds the best suited Movie for one user in every genre.
3. Find what Genre Movies have received the best and worst ratings based on User Rating.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
movies = pd.read_csv("C:/Users/AHSAN RAZA/Desktop/movies.csv")

In [ ]:
print(movies.head())

In [ ]:
movies.columns

In [ ]:
ratings = pd.read_csv("C:/Users/AHSAN RAZA/Desktop/ratings.csv")

In [ ]:
print(ratings.head())

In [ ]:
ratings.columns

In [ ]:
merged_df = pd.merge(ratings, movies, on="movieId", how="inner")
merged_df = merged_df[["movieId", "title", "genres", "userId", "rating", "timestamp"]]
print(merged_df.head())

In [ ]:
merged_df['timestamp'] = merged_df['timestamp'].astype(int)
merged_df['datetime'] = pd.to_datetime(
    merged_df['timestamp'],
    unit='s'
)
merged_df['year'] = merged_df['datetime'].dt.year
merged_df['month'] = merged_df['datetime'].dt.month
merged_df['dayofweek'] = merged_df['datetime'].dt.dayofweek
print(merged_df.head())

In [ ]:
# EDA (Exploratory Data Analysis)

In [ ]:
# 1.) Understand the Data

In [ ]:
print(merged_df.head())
print(merged_df.tail())

In [ ]:
merged_df.shape

In [ ]:
merged_df.columns

In [ ]:
merged_df.describe()

In [ ]:
merged_df.info()

In [ ]:
# 2.) Handle missing values

In [ ]:
merged_df.isnull().sum()

In [ ]:
# 2.a.) Handling missing values for Columns

In [ ]:
thresholds_column = {
    "numeric": 50,     # Drop numeric columns with >50% missing
    "datetime": 30,    # Drop datetime columns with >30% missing
    "object": 60  # Drop object columns with >60% missing
}

In [ ]:
numeric_column = merged_df.select_dtypes(include=["number"])
datetime_column = merged_df.select_dtypes(include=["datetime"])
object_column = merged_df.select_dtypes(include=["object"])

In [ ]:
print("Numeric columns:\n", numeric_column.columns.tolist())
print("Datetime columns:\n", datetime_column.columns.tolist())
print("Object columns:\n", object_column.columns.tolist())

In [ ]:
missing_numeric = numeric_column.isnull().mean() * 100
missing_datetime = datetime_column.isnull().mean() * 100
missing_object = object_column.isnull().mean() * 100

In [ ]:
numeric_column = numeric_column.loc[:, missing_numeric < thresholds_column["numeric"]]
datetime_column = datetime_column.loc[:, missing_datetime < thresholds_column["datetime"]]
object_column = object_column.loc[:, missing_object < thresholds_column["object"]]

In [ ]:
movie_rating = pd.concat([numeric_column, datetime_column, object_column], axis=1)
print(movie_rating)

In [ ]:
# reorder columns
movie_rating = movie_rating[["movieId", "title", "year", "userId", "datetime", "rating"]]
print(movie_rating.head())

In [ ]:
movie_rating.isnull().sum()

In [ ]:
# 2.b.) Handling missing values for Rows

In [ ]:
missing_year_movies = movie_rating[movie_rating['year'].isna()]['movieId'].unique()
for movie in missing_year_movies:
    movie_rows = movie_rating[movie_rating['movieId'] == movie]
    print(f"Movie ID: {movie}")
    print(movie_rows[['movieId', 'title', 'year']])
    print("-" * 40)
both_missing_and_present = []
for movie in missing_year_movies:
    movie_years = movie_rating[movie_rating['movieId'] == movie]['year']
    if movie_years.notna().any() and movie_years.isna().any():
        both_missing_and_present.append(movie)
print("Movies with both missing and present year:", both_missing_and_present)

In [ ]:
movie_rating = movie_rating.dropna(subset=['year'])

In [ ]:
movie_rating.isnull().sum()

In [ ]:
# Univariate analysis

In [ ]:
# Categorical variables
print("Unique Movies:", merged_df['movieId'].nunique())
print("Unique Users:", merged_df['userId'].nunique())
merged_df['title'].value_counts().head(10).plot(kind='bar')
plt.title("Top 10 Most Rated Movies")
plt.xlabel("Movie Title")
plt.ylabel("Number of Ratings")
plt.show()

In [ ]:
# Numerical variables
sns.histplot(merged_df['rating'], bins=10, kde=True)
plt.title("Distribution of Movie Ratings")
plt.show()
sns.boxplot(y=merged_df['rating'])
plt.title("Boxplot of Ratings")
plt.show()
sns.histplot(merged_df['year'], bins=30, kde=True)
plt.title("Distribution of Movie Release Years")
plt.show()

In [ ]:
# Datetime variables

merged_df.groupby(merged_df['datetime'].dt.year)['rating'].count().plot()
plt.title("Number of Ratings per Year")
plt.xlabel("Year")
plt.ylabel("Count")
plt.show()

In [ ]:
# Bivariate Analysis

In [ ]:
# Rating vs Year

sns.scatterplot(data=merged_df, x='year', y='rating', alpha=0.4)
plt.title("Rating vs Movie Release Year")
plt.show()
print("Correlation:", merged_df['year'].corr(merged_df['rating']))

In [ ]:
# Rating vs Movie

top_movies = merged_df.groupby('title')['rating'].mean().sort_values(ascending=False).head(10)
top_movies.plot(kind='bar')
plt.title("Top 10 Highest Rated Movies")
plt.ylabel("Average Rating")
plt.show()

In [ ]:
# Rating vs User Activity

user_activity = merged_df.groupby('userId')['rating'].count()
sns.histplot(user_activity, bins=30, kde=True)
plt.title("User Rating Activity Distribution")
plt.xlabel("Ratings per User")
plt.show()

In [ ]:
# Ratings over Time

merged_df.groupby(merged_df['datetime'].dt.to_period('M'))['rating'].mean().plot()
plt.title("Average Rating Over Time")
plt.xlabel("Time")
plt.ylabel("Average Rating")
plt.show()

In [ ]:
# Multivariate Analysis

In [ ]:
# Correlation Heatmap

numeric_cols = ['rating', 'year']
sns.heatmap(merged_df[numeric_cols].corr(), annot=True, cmap='coolwarm')
plt.title("Correlation Matrix")
plt.show()

In [ ]:
# Ratings by Year and Movie Popularity

movie_stats = merged_df.groupby('title').agg(
    avg_rating=('rating', 'mean'),
    rating_count=('rating', 'count')
).reset_index()

sns.scatterplot(
    data=movie_stats,
    x='rating_count',
    y='avg_rating'
)
plt.title("Movie Popularity vs Average Rating")
plt.xlabel("Number of Ratings")
plt.ylabel("Average Rating")
plt.show()

In [ ]:
# User behaviour over Time

sns.lineplot(
    data=merged_df,
    x=merged_df['datetime'].dt.year,
    y='rating',
    estimator='mean'
)
plt.title("Average Rating by Year")
plt.show()

In [ ]:
# Rating Distribution by Year (Multivariate Boxplot)

merged_df['rating_year'] = merged_df['datetime'].dt.year
sns.boxplot(data=merged_df, x='rating_year', y='rating')
plt.xticks(rotation=90)
plt.title("Ratings Distribution by Year")
plt.show()

In [ ]:
# Recommendation Engine

In [ ]:
# 1.) Most Popular and liked Genre

genre_df = merged_df.copy()
genre_df = genre_df.assign(genres=genre_df['genres'].str.split('|')).explode('genres')
popular_genres = genre_df.groupby('genres').size().sort_values(ascending=False)
print("Most Popular Genres:\n", popular_genres)
liked_genres = genre_df.groupby('genres')['rating'].mean().sort_values(ascending=False)
print("Most Liked Genres:\n", liked_genres)

In [ ]:
# 2.) Best suited movie for One user in every Genre

from surprise import SVD, Dataset, Reader
from surprise.model_selection import train_test_split

reader = Reader(rating_scale=(merged_df['rating'].min(), merged_df['rating'].max()))
data = Dataset.load_from_df(merged_df[['userId','movieId','rating']], reader)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)
algo = SVD()
algo.fit(trainset)

def recommend_top_movie_per_genre(user_id):
    recommendations = {}
    for genre in genre_df['genres'].unique():
        genre_movies = genre_df[genre_df['genres'] == genre]['movieId'].unique()
        preds = [(movie, algo.predict(user_id, movie).est) for movie in genre_movies]
        top_movie_id, top_rating = max(preds, key=lambda x: x[1])
        movie_title = merged_df[merged_df['movieId']==top_movie_id]['title'].iloc[0]
        recommendations[genre] = (movie_title, round(top_rating,2))
    return recommendations

while True:
    try:
        user_id_input = int(input("Enter the User ID for recommendations: "))
        break
    except ValueError:
        print("Please enter a valid integer for User ID.")

top_movies = recommend_top_movie_per_genre(user_id_input)
print(f"\nTop movie recommendations for User {user_id_input} in each genre:")
for genre, (movie, rating) in top_movies.items():
    print(f"{genre}: {movie} (Predicted Rating: {rating})")

In [ ]:
# 3.) Genres with Best and Worst ratings

genre_ratings = genre_df.groupby('genres')['rating'].agg(['mean', 'count']).sort_values('mean', ascending=False)
print("Genres with Best Ratings:\n", genre_ratings.head())
print("Genres with Worst Ratings:\n", genre_ratings.tail())